In [2]:
def load_data(file_name: str) :
    """"
        Helper functions that extracts the data from the input files.
        The format based on which it extracts can be seen at 'knapsack-20.txt'
        The first line holds the nr of total objects n, then n lines represeting 
        the index, value and weight of the values, then the max_weight is on the last line. 
    Args:
        file_name (str): Input file for extracting data

    Returns:
        int : The total number of objects 
        list[tuple[int, int]]: The results consists of the n tuples(represnting (value, weight) pairs)
        int : The maximum value for the total weight
    """
    weights_and_values = []
    with open(file_name) as f:
        total = int(f.readline())
        for _ in range(total) :
            values = f.readline().split()
            n2 = int(values[1])
            n3 = int(values[2])
            weights_and_values.append((n2, n3))
        
        max_value = int(f.readline())
    
    return total, weights_and_values, max_value

In [3]:
def compute_value_for_solution(solution :list, max_value_weight, weights_values:list[tuple]) :
    """For each solution generated it computes the total value accumulated and
    the weight it reaches , if the weight is valid it return the maximum one, otherwise -1.

    Args:
        solution (list): The list of size n representing the made out of 0s and 1s, representing
        whether the values are added or not.
        max_value_value (int): The maximum weight given as input
        weights_values (list[tuple]) : The input data for all the objects (value, weight) pairs
        
    Returns:
    int : Returns the total value if the weight is lower that the max_value_weight,
            otherwise returns -1
    """
    sum_values = 0
    sum_weights = 0
    
    for i in range(len(solution)) :
        sum_values+=solution[i]*weights_values[i][0]
        sum_weights+=solution[i]*weights_values[i][1]
        
    if sum_weights <= max_value_weight :
        return sum_values
    return -1

In [4]:
import random

def crossover(parent1, parent2, crossover_rate) :
    children = []
    
    chance1 = random.random()
    if chance1 <= crossover_rate :
        j = random.randint(0, len(parent1) - 1)
        child1 = parent1[:j] + parent2[j:]
        children.append(child1)
    else :
        children.append(parent1)
        
    chance2 = random.random()
    if chance2 <= crossover_rate :
        j = random.randint(0, len(parent1) - 1)
        child2 = parent2[:j] + parent1[j:]
        children.append(child2)
    else :
        children.append(parent2)

    return children

def mutate(individual, mutation_rate) :
    for i in range(len(individual)) :
        chance = random.random()
        if chance <= mutation_rate :
            individual[i] = 1 - individual[i]
            
    return individual

def tournament_selection(population, fitnesses, tournament_size=3):
    candidates = random.sample(range(len(population)), tournament_size)
    best = max(candidates, key=lambda i: fitnesses[i])
    return population[best]
            

In [11]:
import random
import math

def evolutionary_algorithm_knapsack(filename, iterations, initial_population=30) :
    total, weights_values, max_weight = load_data(filename)
    population = [[random.randint(0,1) for _ in range(total)] for _ in range(initial_population)]
    
    
    for i in range(iterations) :
        
        fitnesses = [compute_value_for_solution(sol, max_weight, weights_values) for sol in population]
        children = []
        
        top_10 = sorted(population,
                        key = lambda x : compute_value_for_solution(x, max_weight, weights_values),
                        reverse=True)
        
        top_10 = top_10[:10]
        
        for i in range((initial_population - 10)//2) :
            parent1 = tournament_selection(population, fitnesses)
            parent2  = tournament_selection(population, fitnesses)
            
            child1, child2 = crossover(parent1, parent2, 0.7)
            mutated_child1 = mutate(child1, 1/total)
            mutated_child2 = mutate(child2, 1/total)
            
            children.append(mutated_child1)
            children.append(mutated_child2)
            
        children+=top_10
        population = children
        
    return compute_value_for_solution(
        max(population, key = lambda x : compute_value_for_solution(x, max_weight, weights_values)),
        max_weight,
        weights_values)
    

In [22]:
evolutionary_algorithm_knapsack('../Lab1Assigment1/knapsack-20.txt', 200, 50)

707

In [25]:
evolutionary_algorithm_knapsack('../Lab1Assigment1/knapsack-200.txt', 10000, 100)

135588

In [ ]:
# 135588 best solutuon found ALL TIME for knapsack 200